In [1]:
import pandas as pd
import os
from datetime import timedelta, datetime

# ── 1. DICCIONARIO: Pasillo Surtido ──────────────────────────────────────────
ruta_pasillo = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 2. PROYECTOS\HANDHELDS SPUE\Pruebas\Abril_Honeywell_Urovo+T1Pro\Pasillo surtido.xlsx"
df_pasillo = pd.read_excel(ruta_pasillo, sheet_name="PS")
dict_loc_pasillo = {
    str(k): v
    for k, v in df_pasillo.drop_duplicates(subset="Clave Localidad")
    .set_index("Clave Localidad")["Pasillo"]
    .to_dict()
    .items()
}
print(f"Localidades cargadas en diccionario: {len(dict_loc_pasillo):,}")

# ── 2. LEER Y COMBINAR ARCHIVOS ──────────────────────────────────────────────
carpeta = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 2. PROYECTOS\HANDHELDS SPUE\Pruebas\Abril_Honeywell_Urovo+T1Pro\Surtido_hist"

dfs = []
for archivo in os.listdir(carpeta):
    ruta = os.path.join(carpeta, archivo)
    if not os.path.isfile(ruta) or not archivo.endswith(".xlsx") or archivo.startswith("~$"):
        continue
    try:
        df_tmp = pd.read_excel(ruta, sheet_name=0)
        dfs.append(df_tmp)
    except Exception as e:
        print(f"Error leyendo {archivo}: {e}")

df = pd.concat(dfs, ignore_index=True)

# Eliminar filas duplicadas (misma fecha+usuario+localidad+código en más de un archivo)
filas_antes = len(df)
df = df.drop_duplicates(subset=["Fecha hora", "Usuario", "Localidad", "Cdigo"])
filas_despues = len(df)
print(f"Filas duplicadas eliminadas: {filas_antes - filas_despues:,} (de {filas_antes:,} → {filas_despues:,})")

# ── 3. TIPOS DE COLUMNAS ─────────────────────────────────────────────────────
df["Fecha hora"] = pd.to_datetime(df["Fecha hora"], dayfirst=True)
for col in ["Usuario", "Anden", "Cdigo", "Localidad"]:
    df[col] = df[col].astype(str)

# ── 4. DÍA OPERATIVO (turno 23:00 – 06:00) ───────────────────────────────────
def dia_operativo(dt):
    if dt.hour < 6:
        return (dt - timedelta(days=1)).strftime("%Y-%m-%d")
    return dt.strftime("%Y-%m-%d")

df["Día Operativo"] = df["Fecha hora"].apply(dia_operativo)
df["Solo Hora"] = df["Fecha hora"].dt.hour
# Turno: scans entre 23:00 y 05:59 → Noche; resto → Dia
df["Turno"] = df["Fecha hora"].apply(lambda dt: "Noche" if dt.hour >= 23 or dt.hour < 6 else "Dia")

# ── 5. MAPEAR PASILLO ────────────────────────────────────────────────────────
df["Pasillo"] = df["Localidad"].map(dict_loc_pasillo)

# ── 6. SEMANA ISO ────────────────────────────────────────────────────────────
df["Semana ISO"] = df["Fecha hora"].dt.isocalendar().week.astype(int)

# ── 7. ORDENAR ───────────────────────────────────────────────────────────────
df = df.sort_values(["Usuario", "Fecha hora"]).reset_index(drop=True)

# ── 8. COMPARACIÓN FILA SIGUIENTE (Visitas) ───────────────────────────────────
df["Next_U"] = df["Usuario"].shift(-1).fillna(df["Usuario"].iloc[-1])
df["Next_L"] = df["Localidad"].shift(-1).fillna(df["Localidad"].iloc[-1])
df["Next_F"] = df["Fecha hora"].shift(-1).ffill()

df["Visita_Val"] = (
    (df["Usuario"] != df["Next_U"]) |
    (df["Localidad"] != df["Next_L"]) |
    ((df["Next_F"] - df["Fecha hora"]) > timedelta(minutes=3))
).astype(int)

# ── 9. MÁXIMOS Y MÍNIMOS POR HORA ─────────────────────────────────────────────
calculo_horas = (
    df.groupby(["Día Operativo", "Turno", "Usuario", "Solo Hora"])
    .agg(V_H=("Visita_Val", "sum"), C_H=("Visita_Val", "count"))
    .reset_index()
)
maxmin_turno = (
    calculo_horas.groupby(["Día Operativo", "Turno", "Usuario"])
    .agg(Esc_Max=("C_H", "max"), Esc_Min=("C_H", "min"), Vis_Max=("V_H", "max"), Vis_Min=("V_H", "min"))
    .reset_index()
)

# ── 10. AGRUPAMIENTO PRINCIPAL POR TURNO ────────────────────────────────────
resumen_turno = (
    df.groupby(["Día Operativo", "Turno", "Usuario"])
    .agg(
        Semana_ISO=("Semana ISO", "first"),
        H_Inicio=("Fecha hora", "min"),
        H_Final=("Fecha hora", "max"),
        Total_Visitas=("Visita_Val", "sum"),
        Total_Codigos=("Fecha hora", "count"),
        Total_Andenes=("Anden", "nunique"),
    )
    .reset_index()
)

# ── 11. PASILLOS TRABAJADOS POR TURNO (como columna en el resumen) ────────────
pasillos_turno = (
    df[df["Pasillo"].notna()]
    .groupby(["Día Operativo", "Turno", "Usuario"])["Pasillo"]
    .apply(lambda x: ", ".join(sorted(x.astype(str).unique())))
    .reset_index()
    .rename(columns={"Pasillo": "Pasillos"})
)

# ── 12. COMBINAR ────────────────────────────────────────────────────────────────
resultado = resumen_turno.merge(maxmin_turno, on=["Día Operativo", "Turno", "Usuario"], how="left")
resultado = resultado.merge(pasillos_turno, on=["Día Operativo", "Turno", "Usuario"], how="left")

# ── 13. MÉTRICAS POR HORA ─────────────────────────────────────────────────────
def calcular_metricas(row):
    h_ini = row["H_Inicio"]
    h_fin = row["H_Final"]
    # Guard: NaT o timestamps invertidos por datos sucios
    if pd.isna(h_ini) or pd.isna(h_fin) or h_fin < h_ini:
        dur_bruta = pd.Timedelta(0)
    else:
        dur_bruta = h_fin - h_ini
    dur_libre = dur_bruta - pd.Timedelta(minutes=30)
    if dur_libre < pd.Timedelta(0):
        dur_libre = pd.Timedelta(0)
    h_dec = dur_libre.total_seconds() / 3600
    total_int = row["Total_Andenes"] + row["Total_Visitas"] + row["Total_Codigos"]
    return pd.Series({
        "Total_Interacciones": total_int,
        "Duracion": dur_libre,
        "Prom_Interacciones/h": total_int / h_dec if h_dec > 0 else 0,
        "Prom_Codigos/h": row["Total_Codigos"] / h_dec if h_dec > 0 else 0,
        "Prom_Visitas/h": row["Total_Visitas"] / h_dec if h_dec > 0 else 0,
        "Prom_Andenes/h": row["Total_Andenes"] / h_dec if h_dec > 0 else 0,
    })

metricas = resultado.apply(calcular_metricas, axis=1)
resultado = pd.concat([resultado, metricas], axis=1)

# ── 14. RENOMBRAR Y REORDENAR COLUMNAS ───────────────────────────────────────
resultado = resultado.rename(columns={
    "Semana_ISO": "Semana ISO",
    "Esc_Max": "Codigos_Max(h)", "Esc_Min": "Codigos_Min(h)",
    "Vis_Max": "Vis_Max(h)", "Vis_Min": "Vis_Min(h)",
})
columnas_finales = [
    "Día Operativo", "Semana ISO", "Usuario", "Turno", "Pasillos",
    "H_Inicio", "H_Final", "Duracion",
    "Total_Interacciones", "Prom_Interacciones/h",
    "Total_Andenes", "Prom_Andenes/h",
    "Total_Visitas", "Prom_Visitas/h",
    "Total_Codigos", "Prom_Codigos/h",
    "Codigos_Max(h)", "Codigos_Min(h)", "Vis_Max(h)", "Vis_Min(h)",
]
resultado = resultado[columnas_finales]

# Formatear Duracion como HH:MM (seguro contra NaT o valores negativos)
resultado["Duracion"] = resultado["Duracion"].apply(
    lambda x: (
        f"{int(x.total_seconds()//3600):02d}:{int((x.total_seconds()%3600)//60):02d}"
        if pd.notna(x) and x.total_seconds() >= 0
        else "00:00"
    )
)

# ── 15. EXPORTAR EXCEL ───────────────────────────────────────────────────────
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ruta_xlsx = rf"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 2. PROYECTOS\HANDHELDS SPUE\Pruebas\Abril_Honeywell_Urovo+T1Pro\resumen_surtido_honeywell_{timestamp}.xlsx"
resultado.to_excel(ruta_xlsx, index=False)
print(f"Excel guardado en: {ruta_xlsx}")

resultado

Localidades cargadas en diccionario: 3,183
Filas duplicadas eliminadas: 0 (de 152,275 → 152,275)
Excel guardado en: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 2. PROYECTOS\HANDHELDS SPUE\Pruebas\Abril_Honeywell_Urovo+T1Pro\resumen_surtido_honeywell_20260507_091200.xlsx


,Día Operativo,Semana ISO,Usuario,Turno,Pasillos,H_Inicio,H_Final,Duracion,Total_Interacciones,Prom_Interacciones/h,Total_Andenes,Prom_Andenes/h,Total_Visitas,Prom_Visitas/h,Total_Codigos,Prom_Codigos/h,Codigos_Max(h),Codigos_Min(h),Vis_Max(h),Vis_Min(h)
0,2026-05-01,18,4002913 4002913 DOMINGUEZ,Noche,"11, 2",2026-05-01 23:25:33,2026-05-02 00:06:30,00:10,126,690.410959,2,10.958904,48,263.013699,76,416.438356,60,16,38,10
1,2026-05-01,18,4003402 4003402 MENA,Noche,"5, 7",2026-05-01 23:19:53,2026-05-02 00:09:32,00:19,145,442.748092,2,6.106870,51,155.725191,92,280.916031,65,27,45,6
2,2026-05-01,18,4006371 4006371 ROMERO,Noche,NaN,2026-05-01 23:42:14,2026-05-02 02:33:42,02:21,37,15.692743,7,2.968897,15,6.361923,15,6.361923,5,2,5,2
3,2026-05-01,18,4009355 4009355 TORRES,Noche,"A1, A2, A3, A4",2026-05-01 23:25:47,2026-05-02 04:33:50,04:38,645,139.183600,13,2.805251,244,52.652401,388,83.725949,92,34,68,25
4,2026-05-01,18,4013635 4013635 PEREZ,Noche,"11, 14, 2, 4, 4A, 5, 6, 8, 9, 9A",2026-05-01 23:21:21,2026-05-02 04:37:21,04:46,763,160.069930,17,3.566434,268,56.223776,478,100.279720,109,36,59,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,2026-05-06,19,980556 PUS0556 CARDIEL,Noche,NaN,2026-05-06 23:37:31,2026-05-07 04:03:26,03:55,11,2.797598,3,0.762981,4,1.017308,4,1.017308,3,1,3,1
423,2026-05-06,19,980635 PUS0635 PAZ,Noche,"C1, C2, C4",2026-05-06 23:12:22,2026-05-07 05:37:07,05:54,1206,203.974630,31,5.243129,475,80.338266,700,118.393235,121,62,89,42
424,2026-05-06,19,980918 PUS0918 ALEGRIA,Noche,2,2026-05-06 23:10:57,2026-05-07 05:12:33,05:31,1127,203.920386,22,3.980700,416,75.271411,689,124.668275,147,9,89,5
425,2026-05-06,19,980969 PUS0969 CANO,Noche,7,2026-05-06 23:29:46,2026-05-07 05:21:45,05:21,675,125.782908,23,4.285936,280,52.176614,372,69.320358,84,11,62,7


In [2]:
# ── DETALLE POR PASILLO: una fila por visita contigua ────────────────────────
# Usar solo filas con Pasillo mapeado, ordenadas por usuario y tiempo
df_cp = df[df["Pasillo"].notna()].copy()
df_cp = df_cp.sort_values(["Usuario", "Día Operativo", "Turno", "Fecha hora"]).reset_index(drop=True)

# Detectar inicio de nueva visita: cambia Pasillo O cambia Anden
# (o cambia usuario/día/turno → siempre nueva visita)
_prev_u  = df_cp["Usuario"].shift(1)
_prev_d  = df_cp["Día Operativo"].shift(1)
_prev_t  = df_cp["Turno"].shift(1)
_prev_p  = df_cp["Pasillo"].shift(1)
_prev_a  = df_cp["Anden"].shift(1)

_nueva_visita = (
    (df_cp["Usuario"]       != _prev_u) |
    (df_cp["Día Operativo"] != _prev_d) |
    (df_cp["Turno"]         != _prev_t) |
    (df_cp["Pasillo"]       != _prev_p) |
    (df_cp["Anden"]         != _prev_a)
)
df_cp["Visita_ID"] = _nueva_visita.cumsum()

# Calcular tiempo por escaneo: gap al siguiente dentro de la misma visita y ≤ 3 min
df_cp["_next_vid"] = df_cp["Visita_ID"].shift(-1)
df_cp["_next_f"]   = df_cp["Fecha hora"].shift(-1).ffill()

_gap      = df_cp["_next_f"] - df_cp["Fecha hora"]
_mismo_v  = df_cp["Visita_ID"] == df_cp["_next_vid"]
_gap_ok   = _gap <= timedelta(minutes=3)

df_cp["Tiempo_Escaneo"] = _gap.where(_mismo_v & _gap_ok, timedelta(0))

# Agrupar por visita
detalle_pasillo = (
    df_cp.groupby(["Día Operativo", "Turno", "Usuario", "Pasillo", "Anden", "Visita_ID"])
    .agg(
        H_Inicio       =("Fecha hora",      "min"),
        H_Final        =("Fecha hora",      "max"),
        Tiempo_Total   =("Tiempo_Escaneo",  "sum"),
        Total_Escaneos =("Cdigo",           "count"),
        Movimientos    =("Localidad",       "nunique"),
    )
    .reset_index()
    .sort_values(["Día Operativo", "Turno", "Usuario", "H_Inicio"])
    .reset_index(drop=True)
    .drop(columns=["Visita_ID"])
)

# Tiempo en formato legible HH:MM:SS
detalle_pasillo["Tiempo_hms"] = detalle_pasillo["Tiempo_Total"].apply(
    lambda x: f"{int(x.total_seconds()//3600):02d}:"
              f"{int((x.total_seconds()%3600)//60):02d}:"
              f"{int(x.total_seconds()%60):02d}"
)

detalle_pasillo = detalle_pasillo[[
    "Día Operativo", "Turno", "Usuario", "Pasillo", "Anden",
    "H_Inicio", "H_Final", "Tiempo_hms",
    "Total_Escaneos", "Movimientos",
]]

# Exportar Excel
ts2 = datetime.now().strftime("%Y%m%d_%H%M%S")
ruta_det = rf"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 2. PROYECTOS\HANDHELDS SPUE\Pruebas\Abril_Honeywell_Urovo+T1Pro\detalle_por_pasillo_{ts2}.xlsx"
detalle_pasillo.to_excel(ruta_det, index=False)
print(f"Excel guardado en: {ruta_det}")

detalle_pasillo

Excel guardado en: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - 2. PROYECTOS\HANDHELDS SPUE\Pruebas\Abril_Honeywell_Urovo+T1Pro\detalle_por_pasillo_20260507_091201.xlsx


,Día Operativo,Turno,Usuario,Pasillo,Anden,H_Inicio,H_Final,Tiempo_hms,Total_Escaneos,Movimientos
0,2026-05-01,Noche,4002913 4002913 DOMINGUEZ,11,9,2026-05-01 23:25:33,2026-05-01 23:40:41,00:11:16,26,18
1,2026-05-01,Noche,4002913 4002913 DOMINGUEZ,2,20,2026-05-01 23:48:05,2026-05-02 00:06:30,00:18:25,50,30
2,2026-05-01,Noche,4003402 4003402 MENA,7,11,2026-05-01 23:19:53,2026-05-01 23:28:02,00:08:09,13,12
3,2026-05-01,Noche,4003402 4003402 MENA,5,16,2026-05-01 23:36:25,2026-05-02 00:09:32,00:33:07,79,39
4,2026-05-01,Noche,4009355 4009355 TORRES,A3,16,2026-05-01 23:25:47,2026-05-01 23:42:31,00:16:44,32,25
...,...,...,...,...,...,...,...,...,...,...
7490,2026-05-06,Noche,980979 PUS0979 SANCHEZ,1,5001,2026-05-07 04:39:53,2026-05-07 04:41:23,00:01:30,6,4
7491,2026-05-06,Noche,980979 PUS0979 SANCHEZ,1,5003,2026-05-07 04:44:55,2026-05-07 04:47:17,00:02:22,16,5
7492,2026-05-06,Noche,980979 PUS0979 SANCHEZ,1A,4,2026-05-07 04:49:35,2026-05-07 04:50:12,00:00:37,5,2
7493,2026-05-06,Noche,980979 PUS0979 SANCHEZ,1,4,2026-05-07 04:51:34,2026-05-07 04:53:33,00:01:59,8,6
